# 13 · k-nearest neighbours

Distance-based baseline on the frozen 84-feature representation. Reads train and validation caches only; no audio, mel cache, or test inputs.


In [1]:
from pathlib import Path
import sys,json,time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); sys.path.insert(0,str(ROOT/'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics,select_f1_threshold,miss_rate_at_false_alarm_rate
cfg=load_project_config(ROOT/'config.yaml'); P=cfg['P']


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Frozen inputs and scaling

The scaler is fit using the owner's Q-D eligibility rule: exclude train rows flagged `silent` or `near_silent`; `crop_all_zero` alone remains eligible.


In [2]:
train=load_tabular(P,'train'); val=load_tabular(P,'val')
Xtr=np.asarray(train['X'],dtype=np.float64); ytr=np.asarray(train['y'],dtype=np.int8)
Xva=np.asarray(val['X'],dtype=np.float64); yva=np.asarray(val['y'],dtype=np.int8); names=np.asarray(train['feature_names']).astype(str)
assert Xtr.shape==(7607,84) and Xva.shape==(1510,84)
assert np.array_equal(train['clip_id'],pd.read_csv(P['splits']/'train.csv',keep_default_na=False).clip_id)
assert np.array_equal(val['clip_id'],pd.read_csv(P['splits']/'val.csv',keep_default_na=False).clip_id)
eligible=~(np.asarray(train['silent'],bool)|np.asarray(train['near_silent'],bool)); assert int(eligible.sum())==7557
scaler=StandardScaler().fit(Xtr[eligible]); Xtr_s=scaler.transform(Xtr); Xva_s=scaler.transform(Xva)


## Prewritten grid

Search `k` ∈ {1, 3, 5, 9, 15, 25, 51}, uniform or distance weights, and Manhattan or Euclidean distance. The baseline is `k=5`, uniform, Euclidean. No class weighting exists; validation selects the score threshold.


In [3]:
ks=[1,3,5,9,15,25,51]; configs=[{'n_neighbors':k,'weights':w,'p':p} for k in ks for w in ('uniform','distance') for p in (1,2)]
default={'n_neighbors':5,'weights':'uniform','p':2}
rows=[]; best_model=None; best_row=None; t0=time.perf_counter()
for i,hp in enumerate(configs):
 model=KNeighborsClassifier(**hp,n_jobs=-1); model.fit(Xtr_s,ytr)
 score=model.predict_proba(Xva_s)[:,1]; threshold=select_f1_threshold(yva,score); m=classification_metrics(yva,score,threshold=threshold)
 row={'config_id':i,**hp,'threshold':float(threshold),'val_f1':m['f1'],'val_roc_auc':m['roc_auc'],
 'false_alarm_rate':m['false_alarm_rate'],'miss_rate':m['miss_rate'],'fit_time_s':0.0}
 rows.append(row)
 if best_row is None or row['val_f1']>best_row['val_f1'] or (row['val_f1']==best_row['val_f1'] and (row['n_neighbors']>best_row['n_neighbors'])):
  best_row=row; best_model=model
results=pd.DataFrame(rows); default_id=next(r['config_id'] for r in rows if r['n_neighbors']==5 and r['weights']=='uniform' and r['p']==2)
base=float(results.loc[results.config_id==default_id,'val_f1'].iloc[0]); results['delta_vs_default']=results.val_f1-base
results['passes_replace_rule']=(results.config_id!=default_id)&(results.delta_vs_default>=.01)
rank=results.sort_values(['val_f1','n_neighbors','weights','p'],ascending=[False,False,True,True]); candidate=rank.iloc[0]
chosen_id=int(candidate.config_id) if (int(candidate.config_id)==default_id or bool(candidate.passes_replace_rule)) else default_id
chosen=results.loc[results.config_id==chosen_id].iloc[0]
model=KNeighborsClassifier(n_neighbors=int(chosen.n_neighbors),weights=chosen.weights,p=int(chosen.p),n_jobs=-1).fit(Xtr_s,ytr)
score=model.predict_proba(Xva_s)[:,1]; threshold=select_f1_threshold(yva,score); metrics=classification_metrics(yva,score,threshold=threshold)
print('Grid fits:',len(results),'| elapsed:',round(time.perf_counter()-t0,2),'s','| candidate:',candidate[['n_neighbors','weights','p','val_f1']].to_dict(),'| frozen:',chosen[['n_neighbors','weights','p']].to_dict())


Grid fits: 28 | elapsed: 41.06 s | candidate: {'n_neighbors': 9, 'weights': 'distance', 'p': 1, 'val_f1': 0.7875647668393783} | frozen: {'n_neighbors': 9, 'weights': 'distance', 'p': 1}


## Diagnostics and freeze

Uniform-neighbour score has at most `k + 1` distinct values, so its ROC curve is stepped. Nearest-neighbour distances compare validation with train as a descriptive shift check.


In [4]:
miss1,thr1=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.01); miss5,thr5=miss_rate_at_false_alarm_rate(yva,score,max_false_alarm_rate=.05)
train_dist=model.kneighbors(Xtr_s,n_neighbors=2,return_distance=True)[0][:,1]
val_dist=model.kneighbors(Xva_s,n_neighbors=1,return_distance=True)[0][:,0]
FIG=ROOT/'results'/'figures'; TABLES=ROOT/'results'/'tables'; MODELS=ROOT/'results'/'models'; CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS): d.mkdir(parents=True,exist_ok=True)
results.to_csv(TABLES/'13_knn_grid.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5)); ax.plot(results.config_id,results.val_f1,'o-'); ax.set(xlabel='Grid configuration',ylabel='Validation F1',title='k-NN grid'); fig.tight_layout(); fig.savefig(FIG/'13_knn_grid.png',dpi=160); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5)); ax.hist(train_dist,bins=40,density=True,alpha=.55,label='Train nearest neighbour'); ax.hist(val_dist,bins=40,density=True,alpha=.55,label='Validation nearest neighbour'); ax.set(xlabel='Standardised Euclidean distance',ylabel='Density',title='Nearest-neighbour distance shift'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/'13_knn_neighbor_distances.png',dpi=160); plt.close(fig)
card={'model':'knn','representation':'84 tabular features','best_hyperparameters':{'n_neighbors':int(chosen.n_neighbors),'weights':chosen.weights,'p':int(chosen.p)},
 'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5)},
 'seeds':[],'train_time_s':float(time.perf_counter()-t0),'diagnostics':{'train_nn_distance_median':float(np.median(train_dist)),'validation_nn_distance_median':float(np.median(val_dist)),'uniform_score_levels_max':int(chosen.n_neighbors)+1},
 'strengths':['nonlinear local decision boundaries','no iterative fitting'], 'weaknesses':['stores full training set','prediction cost grows with train size','sensitive to distance and irrelevant features'], 'decision_log_ids':['D044','D045','D047']}
front_hash=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash']
joblib.dump({'model':model,'scaler':scaler,'feature_names':names.tolist(),'threshold':float(threshold),'threshold_rule':'validation_f1_max_ties_higher','scaler_fit_exclusion':'silent OR near_silent; crop_all_zero alone eligible','front_end_hash':front_hash,'hyperparameters':card['best_hyperparameters']},MODELS/'13_knn.joblib')
(CARDS/'13_knn.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8')
print('Validation:',metrics,'| FAR miss:',miss1,miss5)


Validation: {'accuracy': 0.9456953642384106, 'precision': 0.7562189054726368, 'recall': 0.8216216216216217, 'f1': 0.7875647668393783, 'roc_auc': 0.935577766445691, 'false_alarm_rate': 0.0369811320754717, 'miss_rate': 0.1783783783783784, 'confusion_matrix': [[1276, 49], [33, 152]]} | FAR miss: 0.42162162162162165 0.16216216216216217
